# Сбор данных

In [ ]:
import requests
import time
import pandas as pd
from datetime import datetime

ACCESS_TOKEN = 'your_access_token'
API_VERSION = '5.199'

COMMUNITIES = [ 'arzamas.academy', 'art_and_stories', 'hotel_historian', 'mathhedgehog', 'rgoclub' ]

POSTS_PER_COMMUNITY = 100
SLEEP_TIME = 0.5

In [ ]:
def get_posts(domain, count, token, version, offset=0):
    url = 'https://api.vk.com/method/wall.get'
    params = {
        'access_token': token,
        'v': version,
        'domain': domain,
        'count': min(count, 100),
        'offset': offset,
        'filter': 'owner'
    }
    try:
        response = requests.get(url, params=params)
        data = response.json()
        if 'error' in data:
            return [], 0
        return data.get('response', {}).get('items', []), data.get('response', {}).get('count', 0)
    except:
        return [], 0

def get_comments(owner_id, post_id, token, version):
    url = 'https://api.vk.com/method/wall.getComments'
    all_comments = []
    full_owner_id = int(owner_id) if int(owner_id) < 0 else -int(owner_id)
    offset = 0

    while True:
        params = {
            'access_token': token,
            'v': version,
            'owner_id': full_owner_id,
            'post_id': post_id,
            'count': 100,
            'offset': offset,
            'need_likes': 1
        }
        try:
            response = requests.get(url, params=params)
            data = response.json()
            if 'error' in data:
                break
            items = data.get('response', {}).get('items', [])
            if not items:
                break
            for comment in items:
                all_comments.append({
                    'comment_id': comment.get('id'),
                    'text': comment.get('text', ''),
                    'likes': comment.get('likes', {}).get('count', 0),
                    'date': comment.get('date')
                })
            if len(items) < 100:
                break
            offset += 100
            time.sleep(SLEEP_TIME)
        except:
            break
    return all_comments

def collect_all_data(communities, posts_count, token, version):
    all_data = []

    for community in communities:
        print(f"Сбор {community}...")
        offset = 0
        collected = 0

        while collected < posts_count:
            fetch = min(100, posts_count - collected)
            posts, _ = get_posts(community, fetch, token, version, offset)

            if not posts:
                break

            for post in posts:
                if not post.get('text'):
                    continue

                post_info = {
                    'community': community,
                    'type': 'post',
                    'post_id': post.get('id'),
                    'text': post.get('text', ''),
                    'likes': post.get('likes', {}).get('count', 0),
                    'reposts': post.get('reposts', {}).get('count', 0),
                    'views': post.get('views', {}).get('count', 0),
                    'date': datetime.fromtimestamp(post.get('date')).strftime('%Y-%m-%d %H:%M:%S')
                }
                all_data.append(post_info)

                if post.get('comments', {}).get('count', 0) > 0:
                    comments = get_comments(post.get('owner_id'), post.get('id'), token, version)
                    for comment in comments:
                        comment_info = {
                            'community': community,
                            'type': 'comment',
                            'post_id': post.get('id'),
                            'text': comment.get('text', ''),
                            'likes': comment.get('likes', 0),
                            'reposts': None,
                            'views': None,
                            'date': datetime.fromtimestamp(comment.get('date')).strftime('%Y-%m-%d %H:%M:%S') if comment.get('date') else None
                        }
                        all_data.append(comment_info)
                    time.sleep(SLEEP_TIME)
                    print(f"  + {len(comments)} комментов")
                    time.sleep(SLEEP_TIME)

            collected += len(posts)
            offset += fetch

        print(f"Готово: {len([x for x in all_data if x['community']==community and x['type']=='post'])} постов\n")

    return all_data

In [4]:
dataset = collect_all_data(COMMUNITIES, POSTS_PER_COMMUNITY, ACCESS_TOKEN, API_VERSION)

df = pd.DataFrame(dataset)
df.to_csv('vk_dataset.csv', index=False, encoding='utf-8-sig')

Сбор arzamas.academy...
  + 3 комментов
  + 5 комментов
  + 1 комментов
  + 3 комментов
  + 1 комментов
  + 8 комментов
  + 4 комментов
  + 1 комментов
  + 1 комментов
  + 1 комментов
  + 4 комментов
  + 2 комментов
  + 1 комментов
  + 13 комментов
  + 39 комментов
  + 5 комментов
  + 1 комментов
  + 1 комментов
  + 7 комментов
  + 2 комментов
  + 1 комментов
  + 2 комментов
  + 8 комментов
  + 3 комментов
  + 1 комментов
  + 1 комментов
  + 9 комментов
  + 11 комментов
  + 2 комментов
  + 1 комментов
  + 4 комментов
  + 7 комментов
  + 5 комментов
  + 1 комментов
  + 3 комментов
  + 1 комментов
  + 6 комментов
  + 1 комментов
  + 2 комментов
  + 9 комментов
  + 6 комментов
  + 8 комментов
  + 7 комментов
  + 1 комментов
  + 8 комментов
  + 2 комментов
  + 5 комментов
  + 1 комментов
  + 1 комментов
  + 7 комментов
  + 9 комментов
  + 3 комментов
  + 7 комментов
  + 1 комментов
  + 1 комментов
  + 1 комментов
  + 2 комментов
  + 1 комментов
  + 5 комментов
  + 1 комментов
  + 5 коммент

In [5]:
print(f"\nСохранено: {len(df)} записей (постов: {len(df[df['type']=='post'])}, комментариев: {len(df[df['type']=='comment'])})")


Сохранено: 5743 записей (постов: 484, комментариев: 5259)
